In [ ]:
!pip install -q unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.4/30.4 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 21.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import torch
from google.colab import drive
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import logging
logging.set_verbosity_error()

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/assetmind"
df = pd.read_parquet(f"{BASE}/phmsa_cause.parquet")

model, tok = FastLanguageModel.from_pretrained(
    "unsloth/qwen3-4b-instruct-2507-unsloth-bnb-4bit", max_seq_length=1536, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
==((====))==  Unsloth 2026.10.6: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
CAUSES = {
    "A": ("CORROSION FAILURE", "metal loss from internal or external corrosion, e.g. pitting, wall thinning"),
    "B": ("EQUIPMENT FAILURE", "malfunction or failure of equipment: pumps, valves, seals, gaskets, threaded or "
                               "non-threaded connections, control or relief equipment"),
    "C": ("EXCAVATION DAMAGE", "damage from digging or excavation activity by the operator or a third party"),
    "D": ("INCORRECT OPERATION", "human error: wrong valve position, overfilling, not following procedures, "
                                 "improper installation or operation"),
    "E": ("MATERIAL FAILURE OF PIPE OR WELD", "defects or cracking in the pipe body, seam or weld, from "
                                              "manufacturing, construction or environmental cracking"),
    "F": ("NATURAL FORCE DAMAGE", "earth movement, heavy rain or flooding, lightning, freezing temperatures, high winds"),
    "G": ("OTHER OUTSIDE FORCE DAMAGE", "external damage not from excavation: vehicles, fire or explosion, "
                                        "boats, intentional damage, previous mechanical damage"),
}
cause_list = "\n".join(f"{k}. {name}: {desc}" for k, (name, desc) in CAUSES.items())
LETTER = {name: k for k, (name, _) in CAUSES.items()}

def cause_prompt(text):
    return [{"role": "system", "content": "You are a pipeline integrity engineer classifying incident reports."},
            {"role": "user", "content":
                f"Classify the cause of this pipeline incident into exactly one category.\n\n{cause_list}\n\n"
                "If the narrative contains a supplemental or final update, use the cause stated there.\n\n"
                f"Narrative:\n{text}\n\nAnswer with a single letter (A-G) only."}]

In [ ]:
train = df[(df["split"] == "train") & df["main_track"]]
texts = [tok.apply_chat_template(cause_prompt(t) + [{"role": "assistant", "content": LETTER[l]}], tokenize=False)
         for t, l in zip(train["text"], train["label"])]
ds = Dataset.from_dict({"text": texts})
print(len(ds), "training examples")

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", per_device_train_batch_size=4, gradient_accumulation_steps=4,
        num_train_epochs=1, learning_rate=2e-4, warmup_steps=20, lr_scheduler_type="linear",
        optim="adamw_8bit", weight_decay=0.01, logging_steps=20, seed=42, output_dir="outputs",
        fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(), report_to="none"),
)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
trainer.train()

model.save_pretrained(f"{BASE}/qwen3_phmsa_lora")
tok.save_pretrained(f"{BASE}/qwen3_phmsa_lora")

4557 training examples


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4557 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map:   0%|          | 0/4557 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4557 [00:00<?, ? examples/s]

Unsloth: Removed 11 out of 4557 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
{'loss': '10.14', 'grad_norm': '3.97', 'learning_rate': '0.00018', 'epoch': '0.07036'}
{'loss': '2.037', 'grad_norm': '2.246', 'learning_rate': '0.0001864', 'epoch': '0.1407'}
{'loss': '1.16', 'grad_norm': '2.119', 'learning_rate': '0.0001713', 'epoch': '0.2111'}
{'loss': '0.9046', 'grad_norm': '2.131', 'learning_rate': '0.0001562', 'epoch': '0.2814'}
{'loss': '0.8706', 'grad_norm': '0.696', 'learning_rate': '0.0001411', 'epoch': '0.3518'}
{'loss': '0.8388', 'grad_norm': '0.3131', 'learning_rate': '0.000126', 'epoch': '0.4222'}
{'loss': '0.8475', 'grad_norm': '0.7331', 'learning_rate': '0.0001109', 'epoch': '0.4925'}
{'loss': '0.8342', 'grad_norm': '0.2913', 'learning_rate': '9.585e-05

('/content/drive/MyDrive/assetmind/qwen3_phmsa_lora/tokenizer_config.json',
 '/content/drive/MyDrive/assetmind/qwen3_phmsa_lora/chat_template.jinja',
 '/content/drive/MyDrive/assetmind/qwen3_phmsa_lora/tokenizer.json')